<a href="https://colab.research.google.com/github/Lee-Zan/LangChain/blob/main/6.%20Structured%20Output.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. TypedDict


from typing import TypedDict

class Person(TypedDict):
    name : str
    age : int

new_person : Person = {'name':'Lizan', 'age':'22'}

print(new_person)

{'name': 'Lizan', 'age': '22'}


In [4]:
# 2. With Structured Output TypedDict



from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Annotated, Optional

GOOGLE_API_KEY = "YOUR_GEMINI_API_KEY"

model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=GOOGLE_API_KEY
)

class Review(TypedDict):
    key_themes: Annotated[
        list[str],
        "Write down all key themes discussed in the review as a list"
    ]
    summary: Annotated[
        str,
        "A brief summary of the review"
    ]
    sentiment: Annotated[
        str,
        "Return the sentiment as positive, negative, or neutral"
    ]
    pros: Annotated[
        Optional[list[str]],
        "List all pros mentioned in the review"
    ]
    cons: Annotated[
        Optional[list[str]],
        "List all cons mentioned in the review"
    ]

structured_model = model.with_structured_output(Review)

result = structured_model.invoke("""
I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, it's an absolute powerhouse!

The Snapdragon 8 Gen 3 processor makes everything lightning fast, whether I'm gaming, multitasking, or editing photos.

The 5000mAh battery easily lasts a full day even with heavy use, and the 45W fast charging is a lifesaver.

The S-Pen integration is a great touch for note-taking and quick sketches, though I don't use it often.

What really blew me away is the 200MP camera. The night mode is stunning, capturing crisp, vibrant images even in low light.

Zooming up to 100x actually works well for distant objects, but anything beyond 30x loses quality.

However, the weight and size make it a bit uncomfortable for one-handed use.

Also, Samsung's One UI still comes with bloatware.

The $1,300 price tag is also a hard pill to swallow.

Pros:
- Insanely powerful processor
- Stunning 200MP camera
- Long battery life with fast charging
- S-Pen support

Cons:
- Bulky and heavy
- One UI contains bloatware
- Expensive compared to competitors
""")

print(result)
print("\nSummary:")
print(result["summary"])

print("\nSentiment:")
print(result["sentiment"])

print("\nPros:")
print(result["pros"])

print("\nCons:")
print(result["cons"])

{'key_themes': ['performance', 'battery life', 'camera quality', 'design and ergonomics', 'software', 'price'], 'summary': 'The Samsung Galaxy S24 Ultra is a powerful flagship smartphone featuring a fast Snapdragon 8 Gen 3 processor, excellent battery life, and a stellar 200MP camera. However, it is bulky, expensive, and comes with software bloatware.', 'sentiment': 'mixed', 'pros': ['Insanely powerful processor', 'Stunning 200MP camera', 'Long battery life with fast charging', 'S-Pen support'], 'cons': ['Bulky and heavy', 'One UI contains bloatware', 'Expensive compared to competitors']}

Summary:
The Samsung Galaxy S24 Ultra is a powerful flagship smartphone featuring a fast Snapdragon 8 Gen 3 processor, excellent battery life, and a stellar 200MP camera. However, it is bulky, expensive, and comes with software bloatware.

Sentiment:
mixed

Pros:
['Insanely powerful processor', 'Stunning 200MP camera', 'Long battery life with fast charging', 'S-Pen support']

Cons:
['Bulky and heavy'

In [7]:
# 3. Pydantic Demo



from pydantic import BaseModel, EmailStr, Field
from typing import Optional

class Student(BaseModel):
    name: str = "Lizan Niraula"
    age: Optional[int] = None
    email: EmailStr
    gpa: float = Field(
        default=0.0,
        description="A decimal value representing GPA of the student."
    )

new_student = {
    "age": 22,
    "email": "lizanniraula@gmail.com"
}

student = Student(**new_student)

student_dict = student.model_dump()

print(student_dict["age"])

student_json = student.model_dump_json()

print(student_json)

22
{"name":"Lizan Niraula","age":22,"email":"lizanniraula@gmail.com","gpa":0.0}


In [8]:
# 4. With Structured Output Pydantic




from langchain_google_genai import ChatGoogleGenerativeAI
from typing import Optional, Literal
from pydantic import BaseModel, Field

GOOGLE_API_KEY = "YOUR_GEMINI_API_KEY"

model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=GOOGLE_API_KEY
)

class Review(BaseModel):
    key_themes: list[str] = Field(
        description="Write down all the key themes discussed in the review"
    )

    summary: str = Field(
        description="A brief summary of the review"
    )

    sentiment: Literal["Positive", "Negative"] = Field(
        description="Return the sentiment as either Positive or Negative"
    )

    pros: Optional[list[str]] = Field(
        default=None,
        description="List all pros mentioned in the review"
    )

    cons: Optional[list[str]] = Field(
        default=None,
        description="List all cons mentioned in the review"
    )

    name: Optional[str] = Field(
        default=None,
        description="Name of the reviewer if available"
    )

structured_model = model.with_structured_output(Review)

review = """
I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, it’s an absolute powerhouse!

The Snapdragon 8 Gen 3 processor makes everything lightning fast whether I’m gaming, multitasking, or editing photos.

The 5000mAh battery easily lasts a full day even with heavy use, and the 45W fast charging is a lifesaver.

The S-Pen integration is a great touch for note-taking and quick sketches.

What really blew me away is the 200MP camera. The night mode is stunning, capturing crisp, vibrant images even in low light.

Zooming up to 100x actually works well for distant objects, but anything beyond 30x loses quality.

However, the weight and size make it uncomfortable for one-handed use.

Samsung’s One UI still comes with bloatware.

The $1300 price tag is also a hard pill to swallow.

Pros:
- Powerful processor
- Stunning camera
- Long battery life
- S-Pen support

Cons:
- Heavy and bulky
- One UI contains bloatware
- Expensive
"""

result = structured_model.invoke(review)

print(result)
print("\nSummary:", result.summary)
print("\nSentiment:", result.sentiment)
print("\nThemes:", result.key_themes)
print("\nPros:", result.pros)
print("\nCons:", result.cons)

key_themes=['Performance', 'Battery Life', 'Camera Quality', 'Design and Ergonomics', 'Software', 'Price'] summary="The reviewer is impressed by the Samsung Galaxy S24 Ultra's powerful Snapdragon 8 Gen 3 processor, stunning 200MP camera, long-lasting battery, and S-Pen support, but notes drawbacks such as its heavy and bulky design, software bloatware, and high price tag." sentiment='Positive' pros=['Powerful processor', 'Stunning camera', 'Long battery life', 'S-Pen support'] cons=['Heavy and bulky', 'One UI contains bloatware', 'Expensive'] name=None

Summary: The reviewer is impressed by the Samsung Galaxy S24 Ultra's powerful Snapdragon 8 Gen 3 processor, stunning 200MP camera, long-lasting battery, and S-Pen support, but notes drawbacks such as its heavy and bulky design, software bloatware, and high price tag.

Sentiment: Positive

Themes: ['Performance', 'Battery Life', 'Camera Quality', 'Design and Ergonomics', 'Software', 'Price']

Pros: ['Powerful processor', 'Stunning camera